# 🧠 Daily Challenge: Building Your First Neural Network on MNIST

In this notebook you will:
- Load and preprocess the MNIST handwritten-digit dataset
- Build a fully connected neural network with Keras
- Train, evaluate, and visualize the model
- Run a basic hyperparameter tuning experiment

**Runtime → Change runtime type → T4 GPU** is recommended for faster training.

## 📦 0. Install & Import Dependencies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import confusion_matrix, classification_report

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {tf.config.list_physical_devices("GPU")}')

---
## 1️⃣ Load and Preprocess the MNIST Dataset

In [ ]:
# ── 1.1  Load ────────────────────────────────────────────────────────────────
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print('Raw shapes')
print(f'  X_train : {X_train.shape}   y_train : {y_train.shape}')
print(f'  X_test  : {X_test.shape}    y_test  : {y_test.shape}')

In [ ]:
# ── 1.2  Normalize pixel values to [0, 1] ────────────────────────────────────
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

print(f'Pixel range after normalisation: [{X_train.min()}, {X_train.max()}]')

In [ ]:
# ── 1.3  One-hot encode labels ────────────────────────────────────────────────
NUM_CLASSES = 10

y_train_ohe = to_categorical(y_train, NUM_CLASSES)
y_test_ohe  = to_categorical(y_test,  NUM_CLASSES)

print('Label shapes after one-hot encoding')
print(f'  y_train_ohe : {y_train_ohe.shape}')
print(f'  y_test_ohe  : {y_test_ohe.shape}')
print(f'\nExample – original label 5 → {y_train_ohe[0]}')

In [ ]:
# ── 1.4  Carve out a validation split (10 % of training data) ─────────────────
VAL_SPLIT = 0.1
val_size  = int(len(X_train) * VAL_SPLIT)

X_val, y_val_ohe = X_train[:val_size], y_train_ohe[:val_size]
X_train_fit, y_train_fit = X_train[val_size:], y_train_ohe[val_size:]

print(f'Training   : {X_train_fit.shape[0]:,} samples')
print(f'Validation : {X_val.shape[0]:,} samples')
print(f'Test       : {X_test.shape[0]:,} samples')

In [ ]:
# ── 1.5  Visualise sample images ──────────────────────────────────────────────
fig, axes = plt.subplots(3, 10, figsize=(16, 5))
fig.suptitle('MNIST Sample Images (one per digit class)', fontsize=14, fontweight='bold')

for digit in range(10):
    indices = np.where(y_train == digit)[0][:3]  # first 3 of each class
    for row, idx in enumerate(indices):
        ax = axes[row, digit]
        ax.imshow(X_train[idx], cmap='gray')
        ax.set_title(f'Label: {digit}', fontsize=8)
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── 1.6  Class distribution ───────────────────────────────────────────────────
unique, counts = np.unique(y_train, return_counts=True)

plt.figure(figsize=(8, 4))
plt.bar(unique, counts, color='steelblue', edgecolor='white')
plt.xticks(range(10))
plt.xlabel('Digit Class')
plt.ylabel('Count')
plt.title('Training Set — Class Distribution')
for x, c in zip(unique, counts):
    plt.text(x, c + 50, str(c), ha='center', fontsize=8)
plt.tight_layout()
plt.show()

print('Classes are fairly balanced — no resampling needed.')

---
## 2️⃣ Build a Fully Connected Neural Network

In [ ]:
def build_model(hidden_units_1=128, hidden_units_2=64,
                dropout_rate=0.2, learning_rate=1e-3):
    """
    Fully connected (dense) network for MNIST digit classification.

    Architecture
    ────────────
    Input  : 28 × 28 grayscale image
    Flatten: 784-dimensional vector
    Dense 1: hidden_units_1  neurons, ReLU + Dropout
    Dense 2: hidden_units_2  neurons, ReLU + Dropout
    Output : 10 neurons, Softmax  (one per digit class)
    """
    model = keras.Sequential([
        # ── Input & Flatten ───────────────────────────────────────────────────
        layers.Input(shape=(28, 28)),
        layers.Flatten(),                            # 28*28 → 784

        # ── Hidden Layer 1 ────────────────────────────────────────────────────
        layers.Dense(hidden_units_1, activation='relu'),
        layers.Dropout(dropout_rate),

        # ── Hidden Layer 2 ────────────────────────────────────────────────────
        layers.Dense(hidden_units_2, activation='relu'),
        layers.Dropout(dropout_rate),

        # ── Output Layer ──────────────────────────────────────────────────────
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='MNIST_FCN')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# Build the baseline model
model = build_model()
model.summary()

> **Quick architecture recap**
> | Layer | Output shape | # Params |
> |-------|-------------|----------|
> | Flatten | (None, 784) | 0 |
> | Dense ReLU 1 | (None, 128) | 100,480 |
> | Dropout | (None, 128) | 0 |
> | Dense ReLU 2 | (None, 64) | 8,256 |
> | Dropout | (None, 64) | 0 |
> | Dense Softmax | (None, 10) | 650 |

---
## 3️⃣ Train the Neural Network

In [ ]:
EPOCHS     = 10
BATCH_SIZE = 128

# Early stopping to avoid overfitting
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True, verbose=1
)

history = model.fit(
    X_train_fit, y_train_fit,
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    validation_data = (X_val, y_val_ohe),
    callbacks       = [early_stop],
    verbose         = 1
)

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History', fontsize=14, fontweight='bold')

epochs_ran = range(1, len(history.history['loss']) + 1)

# Loss
ax1.plot(epochs_ran, history.history['loss'],     label='Train Loss',      marker='o')
ax1.plot(epochs_ran, history.history['val_loss'], label='Validation Loss', marker='s', linestyle='--')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Categorical Cross-Entropy Loss')
ax1.set_title('Loss over Epochs')
ax1.legend()
ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(epochs_ran, history.history['accuracy'],     label='Train Accuracy',      marker='o')
ax2.plot(epochs_ran, history.history['val_accuracy'], label='Validation Accuracy', marker='s', linestyle='--')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy over Epochs')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4️⃣ Evaluate the Model's Performance

In [ ]:
# ── 4.1  Test-set accuracy ────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test_ohe, verbose=0)
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc * 100:.2f}%')

In [ ]:
# ── 4.2  Confusion matrix ─────────────────────────────────────────────────────
y_pred_probs = model.predict(X_test, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = y_test   # original integer labels

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=range(10), yticklabels=range(10)
)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label',      fontsize=12)
plt.title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.3  Per-class accuracy & which digits are hardest ───────────────────────
print('Classification Report')
print('─' * 60)
print(classification_report(y_true, y_pred, target_names=[str(i) for i in range(10)]))

per_class_acc = cm.diagonal() / cm.sum(axis=1)
hardest = np.argsort(per_class_acc)[:3]   # 3 digits with lowest accuracy
print(f'\n🔍 Hardest digits to classify (by per-class accuracy):')
for d in hardest:
    print(f'   Digit {d}  →  {per_class_acc[d]*100:.1f}% accuracy')

In [ ]:
# ── 4.4  Visualise misclassified examples ─────────────────────────────────────
misclassified_idx = np.where(y_pred != y_true)[0]
print(f'Total misclassified: {len(misclassified_idx)} / {len(y_true)}')

sample_wrong = misclassified_idx[:20]

fig, axes = plt.subplots(4, 5, figsize=(12, 9))
fig.suptitle('Misclassified Digits (true → predicted)', fontsize=13, fontweight='bold')

for ax, idx in zip(axes.ravel(), sample_wrong):
    ax.imshow(X_test[idx], cmap='gray')
    ax.set_title(f'True:{y_true[idx]}  Pred:{y_pred[idx]}',
                 color='red', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 5️⃣ Bonus — Hyperparameter Tuning Experiment

We compare four configurations to see how hidden-layer size and learning rate affect test accuracy.

In [ ]:
configs = [
    {'hidden_units_1': 64,  'hidden_units_2': 32,  'learning_rate': 1e-3, 'label': 'Small  / LR=1e-3'},
    {'hidden_units_1': 128, 'hidden_units_2': 64,  'learning_rate': 1e-3, 'label': 'Medium / LR=1e-3  (baseline)'},
    {'hidden_units_1': 256, 'hidden_units_2': 128, 'learning_rate': 1e-3, 'label': 'Large  / LR=1e-3'},
    {'hidden_units_1': 128, 'hidden_units_2': 64,  'learning_rate': 1e-4, 'label': 'Medium / LR=1e-4'},
]

results = []

for cfg in configs:
    print(f"\n▶ Training: {cfg['label']}")
    m = build_model(
        hidden_units_1 = cfg['hidden_units_1'],
        hidden_units_2 = cfg['hidden_units_2'],
        learning_rate  = cfg['learning_rate']
    )
    m.fit(
        X_train_fit, y_train_fit,
        epochs          = 10,
        batch_size      = 128,
        validation_data = (X_val, y_val_ohe),
        callbacks       = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                                         restore_best_weights=True)],
        verbose         = 0
    )
    _, acc = m.evaluate(X_test, y_test_ohe, verbose=0)
    results.append({'label': cfg['label'], 'test_acc': acc})
    print(f"  Test accuracy : {acc*100:.2f}%")

In [ ]:
# ── Bar chart of tuning results ───────────────────────────────────────────────
labels  = [r['label']    for r in results]
accs    = [r['test_acc'] for r in results]
colors  = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

plt.figure(figsize=(10, 5))
bars = plt.bar(labels, [a * 100 for a in accs], color=colors, edgecolor='white')
plt.ylim(95, 100)
plt.ylabel('Test Accuracy (%)')
plt.title('Hyperparameter Tuning — Test Accuracy Comparison', fontweight='bold')
plt.xticks(rotation=15, ha='right')

for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.05,
             f'{acc*100:.2f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

best = max(results, key=lambda r: r['test_acc'])
print(f"\n🏆 Best config : {best['label']}  →  {best['test_acc']*100:.2f}%")

---
## ✅ Summary

| Step | What we did |
|------|-------------|
| **Load & preprocess** | Loaded MNIST, normalised pixels to [0, 1], one-hot encoded labels, split into train / val / test |
| **Model** | Flatten → Dense(128, ReLU) → Dropout → Dense(64, ReLU) → Dropout → Dense(10, Softmax) |
| **Loss / Optimiser** | Categorical cross-entropy + Adam |
| **Training** | 10 epochs, batch size 128, early stopping on val_loss |
| **Evaluation** | Test accuracy, confusion matrix, per-class metrics, misclassified examples |
| **Tuning** | Compared 4 configs — larger networks generally help; very low LR slows convergence |

### 💡 Further Experiments to Try
- Add **Batch Normalization** between layers
- Switch to a **CNN** (`Conv2D` + `MaxPooling2D`) — typically reaches 99%+ on MNIST
- Try **learning rate scheduling** (`ReduceLROnPlateau`)
- Use **data augmentation** (`ImageDataGenerator` with small rotations/shifts)